# Prepare the project datasets

Rebuild the processed files from preserved public downloads. The MTA benchmark describes New York demand; the Nairobi files provide separate context. No local waiting-time labels or trained model are included.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts/prepare_data.py').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'scripts/prepare_data.py').is_file(), 'Start the kernel in the workspace or notebooks folder.'
subprocess.run([sys.executable, str(ROOT / 'scripts/prepare_data.py')], check=True, capture_output=True, text=True)
report = json.loads((ROOT / 'reports/data_quality.json').read_text())
report


## Hourly M15 demand

The target is the sum of ridership across fare categories and payment methods. Transfers are already included. Time features are known in advance; same-hour transfers are unavailable for future demand forecasting.


In [ ]:
hourly = pd.read_csv(ROOT / 'data/processed/mta_ridership/m15_route_hourly.csv', parse_dates=['transit_timestamp'])
assert len(hourly) == 2160
assert hourly['transit_timestamp'].is_monotonic_increasing
assert hourly['ridership'].sum() == 1148966
hourly.head()


In [ ]:
hourly.groupby('split', sort=False).agg(rows=('ridership', 'size'), start=('transit_timestamp', 'min'), end=('transit_timestamp', 'max'))


## Nairobi context

Weather covers 2025 in Africa/Nairobi local time. Digital Matatus is a historical 2019 feed. Published GTFS headways are not observed arrival intervals. Keep these files separate from the New York benchmark.


In [ ]:
weather = pd.read_csv(ROOT / 'data/processed/nairobi_weather/hourly_2025.csv')
stops = pd.read_csv(ROOT / 'data/processed/digital_matatus/stops.csv', dtype=str, keep_default_na=False)
assert len(weather) == 8760
assert len(stops) == 4284
{'weather_hours': len(weather), 'historical_stop_records': len(stops)}


## Before training

Fit transformations on the training period, choose models with validation data, and evaluate the test period once. Add lag features using past observations only. Collect local arrivals and boarding observations before evaluating NganyaFlow waiting-time estimates. See `../README.md` for preparation rules and source terms.
